In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_breast_cancer

In [ ]:
data = load_breast_cancer()
data.keys()

In [ ]:
data.target

In [ ]:
data.target_names

In [ ]:
#Train test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(data.data, data.target, test_size=0.33)
N, D = X_train.shape

In [ ]:
#Scale data
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
#Build model
model = nn.Sequential(
    nn.Linear(D, 1),
    nn.Sigmoid())

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters())



In [ ]:
X_train = torch.from_numpy(X_train.astype(np.float32))
X_test = torch.from_numpy(X_test.astype(np.float32))
y_train = torch.from_numpy(y_train.astype(np.float32).reshape(-1,1))
y_test = torch.from_numpy(y_test.astype(np.float32).reshape(-1,1))

In [ ]:
#Train loop

epochs = 1000
train_losses = np.zeros(epochs)
test_losses = np.zeros(epochs)

for i in range(epochs):
    optimizer.zero_grad()
    
    #Forward pass
    output = model(X_train)
    loss = criterion(output, y_train)
    
    #Backward and optimize
    loss.backward()
    optimizer.step()
    
    #Test loss
    output_test = model(X_test)
    loss_test = criterion(output_test, y_test)
    
    #Save losses
    train_losses[i] = loss.item()
    test_losses[i] = loss_test.item()
    
    if (i + 1) % 50 == 0:
        print(f'Epoch {i+1}/{epochs}, Train loss: {loss.item():4f}, Test loss: {loss_test.item():4f}')

In [ ]:
plt.plot(train_losses, label = 'Train label')
plt.plot(test_losses, label = 'Test label')
plt.legend()
plt.show()

In [ ]:
#Check accuracy
with torch.no_grad():
    p_train = model(X_train)
    p_train = np.round(p_train.numpy())
    train_acc = np.mean(y_train.numpy() == p_train)
    
    p_test = model(X_test)
    p_test = np.round(p_test.numpy())
    test_acc = np.mean(y_test.numpy() == p_test)

print(f'Train accuracy: {train_acc}, Test accuracy: {test_acc}')